# AlphaLOB Phase 2 — Notebook 03: Train LOBTransformer (T4 GPU)

**Input:** `/content/lob_features.parquet` (from Notebook 02)

**Output:** `/content/lobster_transformer.onnx` (~25–30 MB)

**Expected runtime:** ~15 minutes on Colab T4 GPU

## Architecture Summary
```
Input: (batch, n_levels=10, features_per_level=4)
       ↓ Linear projection to d_model=64
       ↓ Learned positional encoding (LOB level matters)
       ↓ 6 × Transformer Encoder Layers (8 heads, d_ff=256, dropout=0.1)
       ↓ Global Average Pooling → (batch, 64)
       ↓ Multi-Task Heads:
           Head 1: dir_5s   → Linear(64,3) → softmax (DOWN/NEUTRAL/UP)
           Head 2: dir_30s  → Linear(64,3) → softmax
           Head 3: dir_5min → Linear(64,3) → softmax
           Head 4: spread_compress → Linear(64,1) → sigmoid
           Head 5: vol_imbalance   → Linear(64,1)  (regression)
Loss: Kendall et al. 2018 uncertainty weighting (σₖ learned per task)
```

---

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
# Cell 2: Install dependencies
!pip install torch torchvision polars pyarrow onnx onnxruntime mlflow --quiet

FLASH_ATTN = False
print('✅ Dependencies installed (Skipped flash-attn to save 15 mins of compiling)')

In [ ]:
# Cell 2: Imports and device setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import polars as pl
import numpy as np
import os
import time
import json
import mlflow
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: Hyperparameters
CFG = {
    # Architecture
    'n_levels':     10,
    'features_per_level': 4,   # [price_z, vol_z, wofi_z, kyle_lambda_z]
    'd_model':      64,
    'n_heads':      8,
    'n_layers':     6,
    'd_ff':         256,
    'dropout':      0.1,
    # Training
    'batch_size':   512,
    'epochs':       50,
    'lr':           1e-4,
    'weight_decay': 1e-4,
    'early_stop_patience': 5,
    # Data splits (CHRONOLOGICAL — never shuffle across time!)
    'train_frac':   0.70,
    'val_frac':     0.15,
    'test_frac':    0.15,
    # MLflow
    'experiment':   'AlphaLOB-LOBTransformer',
    # I/O
    'parquet_in':   '/content/drive/MyDrive/AlphaLOB/lob_features.parquet',
    'onnx_out':     '/content/drive/MyDrive/AlphaLOB/lobster_transformer.onnx',
    'seed':         42,
}
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print('✅ Config set')

In [ ]:
# Cell 4: LOB Dataset — builds (n_levels, 4) input tensor per tick
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class LOBDataset(Dataset):
    def __init__(self, df: pl.DataFrame):
        self.n = len(df)
        mid = df['mid_price'].to_numpy()
        n_levels = 10
        X = np.zeros((self.n, n_levels, 4), dtype=np.float32)

        for lvl in range(n_levels):
            bp = df[f'bid_price_{lvl}'].to_numpy()
            ap = df[f'ask_price_{lvl}'].to_numpy()
            bv = df[f'bid_vol_{lvl}'].to_numpy()
            av = df[f'ask_vol_{lvl}'].to_numpy()
            price_dist = ((bp + ap) / 2 - mid) / (mid + 1e-9)
            vol_total  = np.log1p(bv + av)
            wofi = df['wofi_z'].to_numpy()
            kyle = df['kyle_lambda_z'].to_numpy()

            X[:, lvl, 0] = price_dist
            X[:, lvl, 1] = vol_total
            X[:, lvl, 2] = wofi
            X[:, lvl, 3] = kyle

        # FIX #1: Aggressive NaN sanitization for Transformer input
        self.X = torch.tensor(
            np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0),
            dtype=torch.float32
        )

        self.y_5s    = torch.tensor(df['label_5s'].to_numpy(),    dtype=torch.long)
        self.y_30s   = torch.tensor(df['label_30s'].to_numpy(),   dtype=torch.long)
        self.y_5min  = torch.tensor(df['label_5min'].to_numpy(),  dtype=torch.long)

        # FIX #2: Binary target for BCEWithLogitsLoss
        self.y_spread = torch.tensor(
            (df['spread_z'].to_numpy() < 0).astype(np.float32),
            dtype=torch.float32
        )

        # FIX #2b: Sanitize y_imbalance — NaN here crashes MSE loss
        wofi_imb = df['wofi_z'].to_numpy().clip(-3, 3)
        self.y_imbalance = torch.tensor(
            np.nan_to_num(wofi_imb, nan=0.0, posinf=0.0, neginf=0.0),
            dtype=torch.float32
        )

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return (
            self.X[idx], self.y_5s[idx], self.y_30s[idx],
            self.y_5min[idx], self.y_spread[idx], self.y_imbalance[idx]
        )

print('✅ LOBDataset class defined')
\n

In [ ]:
# Cell 5: LOBTransformer Architecture
import torch
import torch.nn as nn
import torch.nn.functional as F

class LOBTransformer(nn.Module):
    def __init__(self, n_levels=10, features_per_level=4, d_model=64,
                 n_heads=8, n_layers=6, d_ff=256, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(features_per_level, d_model)
        self.pos_embedding = nn.Embedding(n_levels, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.n_levels = n_levels

    def forward(self, x):
        x = self.input_proj(x)
        positions = torch.arange(self.n_levels, device=x.device).unsqueeze(0)
        x = x + self.pos_embedding(positions)
        x = self.transformer(x)
        x = self.norm(x)
        return x.mean(dim=1)


class MultiTaskHead(nn.Module):
    def __init__(self, d_model=64, n_direction_classes=2):
        super().__init__()
        self.head_dir_5s   = nn.Sequential(nn.Linear(d_model, 32), nn.GELU(), nn.Linear(32, n_direction_classes))
        self.head_dir_30s  = nn.Sequential(nn.Linear(d_model, 32), nn.GELU(), nn.Linear(32, n_direction_classes))
        self.head_dir_5min = nn.Sequential(nn.Linear(d_model, 32), nn.GELU(), nn.Linear(32, n_direction_classes))
        self.head_spread   = nn.Sequential(nn.Linear(d_model, 16), nn.GELU(), nn.Linear(16, 1))
        self.head_vol      = nn.Sequential(nn.Linear(d_model, 16), nn.GELU(), nn.Linear(16, 1))
        self.log_vars = nn.Parameter(torch.zeros(5))

    def forward(self, z):
        return (
            self.head_dir_5s(z), self.head_dir_30s(z), self.head_dir_5min(z),
            self.head_spread(z).squeeze(-1), self.head_vol(z).squeeze(-1)
        )

    def compute_loss(self, outputs, targets):
        pred_5s, pred_30s, pred_5min, pred_spread, pred_vol = outputs
        y_5s, y_30s, y_5min, y_spread, y_vol = targets

        L1 = F.cross_entropy(pred_5s,   y_5s)
        L2 = F.cross_entropy(pred_30s,  y_30s)
        L3 = F.cross_entropy(pred_5min, y_5min)
        L4 = F.binary_cross_entropy_with_logits(pred_spread, y_spread)
        L5 = F.mse_loss(pred_vol, y_vol)

        losses = torch.stack([L1, L2, L3, L4, L5])

        # FIX #3: Clamp log_vars before exp to prevent NaN explosion
        clamped_log_vars = torch.clamp(self.log_vars, min=-10.0, max=10.0)
        precision = torch.exp(-clamped_log_vars)
        total = (precision * losses + 0.5 * clamped_log_vars).sum()
        return total, losses.detach()

print('✅ LOBTransformer and MultiTaskHead defined')
\n

In [ ]:
# Cell 6: Chronological train/val/test split
import polars as pl

print('Loading features and splitting chronologically...')
df = pl.read_parquet(CFG['parquet_in'])

# Merge raw LOB columns if missing
required_raw = (
    [f'bid_price_{l}' for l in range(10)] + [f'ask_price_{l}' for l in range(10)] +
    [f'bid_vol_{l}'  for l in range(10)] + [f'ask_vol_{l}'  for l in range(10)]
)
missing = [c for c in required_raw if c not in df.columns]
if missing:
    print(f'  ⚠️ {len(missing)} raw LOB columns missing. Merging from lob_data.parquet...')
    raw_path = CFG['parquet_in'].replace('lob_features.parquet', 'lob_data.parquet')
    df_raw = pl.read_parquet(raw_path).head(len(df))
    raw_cols = [c for c in df_raw.columns if c.startswith('bid_') or c.startswith('ask_')]
    df = df.hstack(df_raw.select(raw_cols))

# CRITICAL: Aggressive NaN purge BEFORE creating datasets
# Notebook 02 rolling windows leave NaN in first ~9 rows. fill_nan handles both null and NaN.
df = df.fill_null(0.0).fill_nan(0.0)

if 'spread_z' not in df.columns:
    df = df.with_columns(pl.lit(0.0).alias('spread_z'))

n = len(df)
n_train = int(n * CFG['train_frac'])
n_val   = int(n * CFG['val_frac'])

df_train = df.slice(0, n_train)
df_val   = df.slice(n_train, n_val)
df_test  = df.slice(n_train + n_val, n - n_train - n_val)

print(f'  Total: {n:,} | Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')

print('Building datasets...')
train_ds = LOBDataset(df_train)
val_ds   = LOBDataset(df_val)
test_ds  = LOBDataset(df_test)

# num_workers=0 is REQUIRED in Colab to prevent multiprocessing crashes
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=0, pin_memory=False)

print(f'✅ DataLoaders ready | Batches per epoch: {len(train_loader):,}')
\n

In [ ]:
# Cell 7: Training loop
import time
import mlflow
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

encoder = LOBTransformer(
    n_levels=CFG['n_levels'], features_per_level=CFG['features_per_level'],
    d_model=CFG['d_model'], n_heads=CFG['n_heads'], n_layers=CFG['n_layers'],
    d_ff=CFG['d_ff'], dropout=CFG['dropout']
).to(device)

head = MultiTaskHead(d_model=CFG['d_model']).to(device)

optimizer = AdamW(list(encoder.parameters()) + list(head.parameters()),
                  lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'])
scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

mlflow.set_experiment(CFG['experiment'])

history = {'train_loss': [], 'val_loss': [], 'val_acc_30s': []}
best_val_loss = float('inf')
patience_counter = 0

print(f'Starting training for {CFG["epochs"]} epochs on {device}')
total_start = time.time()

with mlflow.start_run(run_name='LOBTransformer-v1'):
    mlflow.log_params(CFG)

    for epoch in range(1, CFG['epochs'] + 1):
        encoder.train(); head.train()
        train_loss = 0.0
        t_ep = time.time()

        for batch in tqdm(train_loader, desc=f'Epoch {epoch:2d}', leave=False):
            X, y5s, y30s, y5min, y_spr, y_vol = [b.to(device) for b in batch]
            optimizer.zero_grad()

            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                z = encoder(X)
                out = head(z)
                loss, _ = head.compute_loss(out, (y5s, y30s, y5min, y_spr, y_vol))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(head.parameters()), 1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        encoder.eval(); head.eval()
        val_loss = 0.0
        correct_30s = 0; total_30s = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc='Val', leave=False):
                X, y5s, y30s, y5min, y_spr, y_vol = [b.to(device) for b in batch]
                with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                    z = encoder(X)
                    out = head(z)
                    loss, _ = head.compute_loss(out, (y5s, y30s, y5min, y_spr, y_vol))
                val_loss += loss.item()
                preds_30s = out[1].argmax(dim=1)
                correct_30s += (preds_30s == y30s).sum().item()
                total_30s   += len(y30s)

        val_loss /= len(val_loader)
        val_acc_30s = correct_30s / total_30s
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc_30s'].append(val_acc_30s)

        mlflow.log_metrics({'train_loss': train_loss, 'val_loss': val_loss,
                           'val_acc_30s': val_acc_30s, 'lr': scheduler.get_last_lr()[0]}, step=epoch)

        ep_time = time.time() - t_ep
        print(f'Epoch {epoch:3d}/{CFG["epochs"]} | Train: {train_loss:.4f} | '
              f'Val: {val_loss:.4f} | Val Acc 30s: {val_acc_30s*100:.2f}% | '
              f'LR: {scheduler.get_last_lr()[0]:.2e} | {ep_time:.0f}s')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({'encoder': encoder.state_dict(), 'head': head.state_dict()},
                       '/content/drive/MyDrive/AlphaLOB/best_checkpoint.pt')
        else:
            patience_counter += 1
            if patience_counter >= CFG['early_stop_patience']:
                print(f'⏹️ Early stopping at epoch {epoch}')
                break

total_time = time.time() - total_start
print(f'✅ Training complete in {total_time/60:.1f} minutes')
print(f'   Best val loss: {best_val_loss:.4f}')
print(f'   Best val acc (30s): {max(history["val_acc_30s"])*100:.2f}%')
\n

In [ ]:
# Cell 8: Plot training curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LOBTransformer Training Curves', fontsize=13, fontweight='bold')

epochs_ran = len(history['train_loss'])
x = range(1, epochs_ran + 1)

ax1.plot(x, history['train_loss'], label='Train Loss', color='#2196F3')
ax1.plot(x, history['val_loss'],   label='Val Loss',   color='#F44336')
ax1.set_title('Loss (Kendall Multi-Task)')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(x, [a*100 for a in history['val_acc_30s']], color='#4CAF50', linewidth=2)
ax2.axhline(50, color='red', linestyle='--', alpha=0.7, label='Random baseline (50%)')
ax2.axhline(58.2, color='orange', linestyle='--', alpha=0.7, label='Target (58.2%)')
ax2.set_title('30s Directional Accuracy (Validation)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy %')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved to /content/training_curves.png')

In [ ]:
# Cell 9: Load best checkpoint and evaluate on held-out TEST set

checkpoint = torch.load('/content/best_checkpoint.pt', map_location=device)
encoder.load_state_dict(checkpoint['encoder'])
head.load_state_dict(checkpoint['head'])
encoder.eval(); head.eval()

all_preds_30s = []; all_true_30s = []
all_preds_5s  = []; all_true_5s  = []

with torch.no_grad():
    for batch in test_loader:
        X, y5s, y30s, y5min, y_spr, y_vol = [b.to(device) for b in batch]
        z   = encoder(X)
        out = head(z)
        all_preds_30s.extend(out[1].argmax(dim=1).cpu().numpy())
        all_true_30s.extend(y30s.cpu().numpy())
        all_preds_5s.extend(out[0].argmax(dim=1).cpu().numpy())
        all_true_5s.extend(y5s.cpu().numpy())

acc_30s = np.mean(np.array(all_preds_30s) == np.array(all_true_30s))
acc_5s  = np.mean(np.array(all_preds_5s)  == np.array(all_true_5s))

print('=== TEST SET RESULTS ===')
print(f'  5s  directional accuracy: {acc_5s*100:.2f}%')
print(f'  30s directional accuracy: {acc_30s*100:.2f}%  ← KEY METRIC (target: 58.2%)')
print()
if acc_30s >= 0.55:
    print('✅ Model shows genuine predictive edge (>55% accuracy)')
elif acc_30s >= 0.52:
    print('⚠️  Model has modest edge. Consider: more data, longer training, or hyperparameter tuning')
else:
    print('❌ Model near random. Check: feature engineering, data quality, look-ahead bias')

In [ ]:
# Cell 10: Export to ONNX
# This is run in Notebook 06 but we do a quick export check here too

import onnx
import onnxruntime as ort

# Create a combined model for ONNX export
class LOBModel(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head    = head

    def forward(self, x):
        z = self.encoder(x)
        d5s, d30s, d5min, spread, vol = self.head(z)
        return (
            torch.softmax(d5s, dim=-1),
            torch.softmax(d30s, dim=-1),
            torch.softmax(d5min, dim=-1),
            torch.sigmoid(spread),
            vol
        )

combined = LOBModel(encoder.cpu(), head.cpu())
combined.eval()

dummy_input = torch.randn(1, CFG['n_levels'], CFG['features_per_level'])

torch.onnx.export(
    combined,
    dummy_input,
    CFG['onnx_out'],
    input_names=['lob_snapshot'],
    output_names=['dir_5s', 'dir_30s', 'dir_5min', 'spread_compress', 'vol_imbalance'],
    dynamic_axes={'lob_snapshot': {0: 'batch_size'}},
    opset_version=17,
    export_params=True
)

# Verify shape and latency
sess = ort.InferenceSession(CFG['onnx_out'],
                            providers=['CPUExecutionProvider'])
test_in = {'lob_snapshot': dummy_input.numpy()}
outputs = sess.run(None, test_in)

assert outputs[0].shape == (1, 2), f'dir_5s shape wrong: {outputs[0].shape}'
assert outputs[1].shape == (1, 2), f'dir_30s shape wrong: {outputs[1].shape}'

import time as _time
times = []
for _ in range(200):
    t0 = _time.perf_counter()
    sess.run(None, test_in)
    times.append((_time.perf_counter() - t0) * 1000)

file_mb = os.path.getsize(CFG['onnx_out']) / 1e6
print(f'✅ ONNX model saved: {CFG["onnx_out"]} ({file_mb:.1f} MB)')
print(f'   p50 latency: {np.percentile(times, 50):.1f}ms')
print(f'   p99 latency: {np.percentile(times, 99):.1f}ms   ← must be < 15ms')
print('
✅ NOTEBOOK 03 COMPLETE')
print('   Next step → Run 04_train_regimehmm.ipynb')